# 03 Feature Engineering

This notebook transforms the cleaned Bank Marketing dataset into model-ready features.

In the MLOps lifecycle, feature engineering is the bridge between business data and model training. The goal is not just to transform columns technically, but to make relevant patterns usable for a model while avoiding data leakage.

## Feature Engineering Goals

This notebook demonstrates:

- Separating identifiers, target columns, numeric features, and categorical features. 
- Creating a train/test split before fitting preprocessing transformations.
- Building a scikit-learn `ColumnTransformer`. 
- Imputing and scaling numeric features. 
- Imputing and one-hot encoding categorical features. 
- Exporting processed train/test data for the next notebook.

We still keep all logic in the notebook. Refactoring into `src/` comes later. 

## Leakage Reminder

Feature transformations can leak information if they are fitted on the full dataset before the train/test split.

For example:

- Imputation values should be learned only from the training data.
- Scaling parameters should be learned only from the training data.
- Encoded category structure should be created from the training data and then applied to test data.

Therefore, the dataset is split first, and the preprocessing pipeline is fitted only on X_train.

## 1. Setup

In [3]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer #Permite aplicar transformaciones diferentes según el tipo de columna.
from sklearn.impute import SimpleImputer #Imputer = rellenador de datos faltantes.
from sklearn.model_selection import train_test_split #divide Train/Test.
from sklearn.pipeline import Pipeline #"Haz estos pasos siempre en este orden."
from sklearn.preprocessing import OneHotEncoder, StandardScaler #Hace que todas las columnas numéricas tengan una escala parecida.

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

RANDOM_STATE = 42
TEST_SIZE = 0.2


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "interim" / "bank_marketing_cleaned.csv").exists():
            return path
    raise FileNotFoundError("Could not find project root with data/interim/bank_marketing_cleaned.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "bank_marketing_cleaned.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned data path: {INTERIM_DATA_PATH}")

Project root: /home/patri/master/ads2-bank-marketing-project
Cleaned data path: /home/patri/master/ads2-bank-marketing-project/data/interim/bank_marketing_cleaned.csv


### New Concepts

- **train_test_split** → Splits the dataset into training and test sets.
- **SimpleImputer** → Fills missing values.
- **StandardScaler** → Scales numerical features.
- **OneHotEncoder** → Encodes categorical variables.
- **Pipeline** → Executes preprocessing steps in order.
- **ColumnTransformer** → Applies different pipelines to different column types.

## 2. Load Cleaned Data

In [4]:
cleaned_df = pd.read_csv(INTERIM_DATA_PATH)

print(f"Rows: {cleaned_df.shape[0]:,}")
print(f"Columns: {cleaned_df.shape[1]:,}")
cleaned_df.head()

Rows: 45,211
Columns: 17


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,y_binary
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,-1,0,unknown,no,0


In [5]:
cleaned_df.dtypes.rename("dtype").to_frame()

,dtype
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64


#### Key Observations

- Cleaned dataset loaded successfully.
- The target has been encoded as `y_binary`.
- The leakage feature `duration` has been removed.
- The dataset is ready for preprocessing.

## Imputation Reflection

- Missing values should not be imputed blindly.
- The reason for missing data must be understood first.
- The same imputation strategy must be used during training and inference.
- This is not only a technical decision but a business decision too.

## 3. Create Business Features

A small number of business features are created to improve interpretability.

These features summarize relevant customer information while avoiding data leakage.

In [6]:
def create_business_features(df):

    featured = df.copy()

    featured["PreviouslyContacted"] = (
        featured["pdays"] != -1
    ).astype("int64")

    featured["HasAnyLoan"] = (
        (featured["housing"] == "yes") |
        (featured["loan"] == "yes")
    ).astype("int64")

    featured["BalanceGroup"] = pd.cut(
        featured["balance"],
        bins=[float("-inf"), 0, 1000, 5000, float("inf")],
        labels=["Negative", "Low", "Medium", "High"],
    ).astype("object")

    return featured

In [8]:
featured_df = create_business_features(cleaned_df)

featured_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,y_binary,PreviouslyContacted,HasAnyLoan,BalanceGroup
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,0,0,1,Medium
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,0,0,1,Low
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,0,0,1,Low
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,0,0,1,Medium
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,-1,0,unknown,no,0,0,0,Low


#### Key Observations

- `PreviouslyContacted` indicates whether the client participated in a previous campaign.
- `HasAnyLoan` combines housing and personal loan information.
- `BalanceGroup` converts account balance into interpretable business categories.
- The original variables are preserved alongside the engineered features.
- No feature uses information that would be unavailable before contact.

### Feature Logic at Inference Time

New customer records must pass through the same feature creation logic used during training.

The following features must be calculated consistently:

1. `PreviouslyContacted` from `pdays`.
2. `HasAnyLoan` from `housing` and `loan`.
3. `BalanceGroup` from `balance`.

The feature creation function and the fitted preprocessing pipeline must therefore be reused together during prediction.

In [9]:
engineered_features = [
    "PreviouslyContacted",
    "HasAnyLoan",
    "BalanceGroup",
]

featured_df[engineered_features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PreviouslyContacted,45211.0,NaN,NaN,NaN,0.182633,0.386369,0.0,0.0,0.0,0.0,1.0
HasAnyLoan,45211.0,NaN,NaN,NaN,0.619473,0.485522,0.0,0.0,1.0,1.0,1.0
BalanceGroup,45211,4,Low,23300,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Feature Design Notes

The engineered features are simple, interpretable, and available before prediction.

They summarize customer information without introducing data leakage:

- PreviouslyContacted
- HasAnyLoan
- BalanceGroup

Feature importance will be evaluated after model training.

## 4. Define Feature Groups

In [10]:

TARGET_COLUMNS = ["y", "y_binary"]

NUMERIC_FEATURES = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

BINARY_FEATURES = [
    "PreviouslyContacted",
    "HasAnyLoan",
]

CATEGORICAL_FEATURES = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
    "BalanceGroup",
]

FEATURE_COLUMNS = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

feature_overview = pd.DataFrame({
    "group": (
        ["numeric_continuous"] * len(NUMERIC_FEATURES)
        + ["binary_indicator"] * len(BINARY_FEATURES)
        + ["categorical"] * len(CATEGORICAL_FEATURES)
    ),
    "column": FEATURE_COLUMNS,
})

feature_overview

,group,column
0,numeric_continuous,age
1,numeric_continuous,balance
2,numeric_continuous,day
3,numeric_continuous,campaign
4,numeric_continuous,pdays
5,numeric_continuous,previous
6,binary_indicator,PreviouslyContacted
7,binary_indicator,HasAnyLoan
8,categorical,job
9,categorical,marital


### Feature Groups

Columns are grouped according to the preprocessing they require.

- Numerical features will be imputed and scaled.
- Binary features will be kept as numeric indicators.
- Categorical features will be one-hot encoded.

## Feature Selection Note: `balance` and `BalanceGroup`

BalanceGroup is derived directly from balance, so both features contain related information.

Keeping both can still be useful in an early baseline:

- `balance` preserves the exact account balance.
- `BalanceGroup` provides broader financial segments.

Later model evaluation can determine whether both features should be kept.

In [11]:
missing_features = set(FEATURE_COLUMNS + TARGET_COLUMNS) - set(featured_df.columns)
if missing_features:
    raise ValueError(f"Missing expected columns: {sorted(missing_features)}")

print(f"Number of model features before encoding: {len(FEATURE_COLUMNS)}")
print(f"Target columns excluded from features: {TARGET_COLUMNS}")

Number of model features before encoding: 18
Target columns excluded from features: ['y', 'y_binary']


#### Key Observations

- Features are grouped by preprocessing requirements.
- Engineered features are available before prediction.
- Target columns are excluded from model features.
- All expected columns were successfully validated.

## 5. Create Features and Target (X and Y)

The dataset is separated into:

- **X**: input features used for prediction.
- **y**: target variable the model will learn to predict.

Target columns are excluded from `X` to avoid data leakage.

In [12]:
X = featured_df[FEATURE_COLUMNS].copy()
y = featured_df["y_binary"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
y.value_counts(normalize=True).mul(100).round(2).rename("target_share_percent")

X shape: (45211, 18)
y shape: (45211,)


y_binary
0    88.3
1    11.7
Name: target_share_percent, dtype: float64

## 6. Train/Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (36168, 18)
X_test:  (9043, 18)
y_train: (36168,)
y_test:  (9043,)


In [14]:
target_split_check = pd.DataFrame({
    "train_percent": y_train.value_counts(normalize=True).mul(100).round(2),
    "test_percent": y_test.value_counts(normalize=True).mul(100).round(2),
})

target_split_check

,train_percent,test_percent
y_binary,,
0,88.3,88.3
1,11.7,11.7


#### Train/Test Split

- 80% of the data is used for training.
- 20% is reserved for testing.
- `random_state=42` makes the split reproducible.
- `stratify=y` preserves the target class distribution in both sets.
- The test set remains unseen during training.

## 7. Build Preprocessing Pipeline

A preprocessing pipeline is defined for each feature group.

- Numerical: median imputation + standard scaling.
- Binary: most frequent imputation.
- Categorical: most frequent imputation + one-hot encoding.

The preprocessing steps are defined here but are not fitted yet.

In [16]:
def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("binary", binary_pipeline, BINARY_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )


preprocessor = build_preprocessor()
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

## 8. Fit on Training Data and Transform Both Splits

In [17]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"Processed X_test shape:  {X_test_processed.shape}")

Processed X_train shape: (36168, 56)
Processed X_test shape:  (9043, 56)


In [18]:
X_train_processed

array([[-0.46043404, -0.16441038,  1.58212355, ...,  1.        ,
         0.        ,  0.        ],
       [-1.58964093,  0.89962705, -1.2983841 , ...,  0.        ,
         1.        ,  0.        ],
       [ 0.29237054, -0.36548575, -0.45823603, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 0.38647112, -0.41925793,  1.10203894, ...,  1.        ,
         0.        ,  0.        ],
       [-1.3073392 ,  0.38895426, -1.17836295, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.79797972, -0.28922702,  0.38191203, ...,  1.        ,
         0.        ,  0.        ]], shape=(36168, 56))

In [19]:
feature_names = preprocessor.get_feature_names_out()

processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)
processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index,
)

processed_train_df.head()

,numeric__age,numeric__balance,numeric__day,numeric__campaign,numeric__pdays,numeric__previous,binary__PreviouslyContacted,binary__HasAnyLoan,categorical__job_admin.,categorical__job_blue-collar,categorical__job_entrepreneur,categorical__job_housemaid,categorical__job_management,categorical__job_retired,categorical__job_self-employed,categorical__job_services,categorical__job_student,categorical__job_technician,categorical__job_unemployed,categorical__job_unknown,categorical__marital_divorced,categorical__marital_married,categorical__marital_single,categorical__education_primary,categorical__education_secondary,categorical__education_tertiary,categorical__education_unknown,categorical__default_no,categorical__default_yes,categorical__housing_no,categorical__housing_yes,categorical__loan_no,categorical__loan_yes,categorical__contact_cellular,categorical__contact_telephone,categorical__contact_unknown,categorical__month_apr,categorical__month_aug,categorical__month_dec,categorical__month_feb,categorical__month_jan,categorical__month_jul,categorical__month_jun,categorical__month_mar,categorical__month_may,categorical__month_nov,categorical__month_oct,categorical__month_sep,categorical__poutcome_failure,categorical__poutcome_other,categorical__poutcome_success,categorical__poutcome_unknown,categorical__BalanceGroup_High,categorical__BalanceGroup_Low,categorical__BalanceGroup_Medium,categorical__BalanceGroup_Negative
24001,-0.460434,-0.164410,1.582124,-0.246104,-0.410910,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
43409,-1.589641,0.899627,-1.298384,0.398202,1.446096,2.664584,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
20669,0.292371,-0.365486,-0.458236,0.398202,-0.410910,-0.241509,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
18810,0.668773,-0.445003,1.822166,2.653271,-0.410910,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
23130,-0.272233,-0.361249,1.222060,2.331118,-0.410910,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


#### Fit and Transform

The preprocessor is fitted only on the training data.

- fit_transform(X_train): learns preprocessing parameters and transforms the training set.
- transform(X_test): applies the same learned transformations to the test set.

This prevents data leakage because no information from the test set is used during preprocessing.

One-Hot Encoding increases the number of features from 18 to 56.

In [20]:
pd.DataFrame({
    "feature_name": feature_names,
}).head(40)

,feature_name
0,numeric__age
1,numeric__balance
2,numeric__day
3,numeric__campaign
4,numeric__pdays
5,numeric__previous
6,binary__PreviouslyContacted
7,binary__HasAnyLoan
8,categorical__job_admin.
9,categorical__job_blue-collar


#### Key Observations

- The preprocessor was fitted only on the training data.
- The same transformations were applied to the test data.
- Data leakage was avoided during preprocessing.
- One-Hot Encoding expanded the feature space from 18 to 56 features.
- The processed datasets are ready for model training.

## 9. Validate Processed Features

The processed datasets are validated before model training.

The validation confirms that:

- No missing values remain.
- The expected number of rows is preserved.
- The preprocessing pipeline generated 56 model features.
- Numerical features were successfully scaled.
- Categorical features were successfully one-hot encoded.

In [22]:
validation = {
    "train_missing_values": int(processed_train_df.isna().sum().sum()),
    "test_missing_values": int(processed_test_df.isna().sum().sum()),
    "train_rows": processed_train_df.shape[0],
    "test_rows": processed_test_df.shape[0],
    "processed_features": processed_train_df.shape[1],
}

validation

{'train_missing_values': 0,
 'test_missing_values': 0,
 'train_rows': 36168,
 'test_rows': 9043,
 'processed_features': 56}

In [23]:
processed_train_df.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
numeric__age,36168.0,-1.194454e-16,1.000014,-2.154244,-0.742736,-0.178132,0.668773,5.091500
numeric__balance,36168.0,-2.043144e-17,1.000014,-3.058331,-0.420887,-0.298026,0.021104,32.837370
numeric__day,36168.0,8.801237e-17,1.000014,-1.778469,-0.938321,0.021849,0.621954,1.822166
numeric__campaign,36168.0,4.557784e-17,1.000014,-0.568256,-0.568256,-0.246104,0.076049,19.405212
numeric__pdays,36168.0,2.514639e-17,1.000014,-0.410910,-0.410910,-0.410910,-0.410910,8.295054
numeric__previous,36168.0,-1.493067e-17,1.000014,-0.241509,-0.241509,-0.241509,-0.241509,113.926433
binary__PreviouslyContacted,36168.0,1.820394e-01,0.385882,0.000000,0.000000,0.000000,0.000000,1.000000
binary__HasAnyLoan,36168.0,6.211568e-01,0.485106,0.000000,0.000000,1.000000,1.000000,1.000000
categorical__job_admin.,36168.0,1.144935e-01,0.318414,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__job_blue-collar,36168.0,2.164897e-01,0.411858,0.000000,0.000000,0.000000,0.000000,1.000000


#### Key Observations

- No missing values remain after preprocessing.
- The processed datasets contain 56 model features.
- Numerical and categorical transformations were applied successfully.
- The processed datasets are ready for model training.

## 10. Save Processed Data for the Baseline Notebook

In [24]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_train_df.to_csv(PROCESSED_DIR / "X_train_processed.csv",index=False,)
processed_test_df.to_csv(PROCESSED_DIR / "X_test_processed.csv",index=False,)
y_train.to_frame("y_binary").to_csv(PROCESSED_DIR / "y_train.csv",index=False,)
y_test.to_frame("y_binary").to_csv(PROCESSED_DIR / "y_test.csv",index=False,)

pd.Series(feature_names,name="feature_name",).to_csv(PROCESSED_DIR / "feature_names.csv",index=False,)

metadata_columns = [
    "age",
    "job",
    "marital",
    "education",
    "balance",
    "housing",
    "loan",
    "contact",
    "month",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "y",
    "y_binary",
    "PreviouslyContacted",
    "HasAnyLoan",
    "BalanceGroup",
]

train_metadata = featured_df.loc[X_train.index,metadata_columns,].copy()
test_metadata = featured_df.loc[X_test.index,metadata_columns,].copy()
train_metadata.insert(0,"row_id",range(len(train_metadata)),)
test_metadata.insert(0,"row_id",range(len(test_metadata)),)
train_metadata.to_csv(PROCESSED_DIR / "train_metadata.csv",index=False,)
test_metadata.to_csv(PROCESSED_DIR / "test_metadata.csv",index=False,)

print(f"Saved processed data to: {PROCESSED_DIR}")
print("Saved train/test metadata for traceability and error slicing.")

Saved processed data to: /home/patri/master/ads2-bank-marketing-project/data/processed
Saved train/test metadata for traceability and error slicing.


### Save Processed Data

The processed training and test datasets are saved for the baseline modeling notebook.

Metadata is also preserved to support traceability and later error slicing.

## Reflection: Pipeline Robustness

The preprocessing pipeline handles several common situations:

- Missing values through imputation.
- Previously unseen categories with `OneHotEncoder(handle_unknown="ignore")`.
- Consistent feature scaling.

However, it does not validate the raw input schema.

A production API should additionally verify:

- required columns;
- expected data types;
- valid categorical values;
- business validation rules.

Input validation belongs to the prediction service, not to the preprocessing pipeline itself.

## Feature Engineering Summary

This notebook prepared the data for baseline model training.

Main steps completed:

- Created three business features:
  - `PreviouslyContacted`
  - `HasAnyLoan`
  - `BalanceGroup`
- Split the dataset into training and test sets.
- Built a preprocessing pipeline using `ColumnTransformer`.
- Imputed missing values where required.
- Scaled numerical features.
- One-hot encoded categorical features.
- Expanded the feature space from 18 to 56 model features.
- Saved processed datasets for the baseline notebook.
- Saved metadata for traceability and future error analysis.

The next notebook will train and evaluate baseline machine learning models.